# Chapter 4 &mdash; DFA as String Classifiers

**Concept 8 of the Chapter 4 decomposition:** *DFA as String Classifiers: Partitioning $\Sigma^*$*

Every machine in this book partitions $\Sigma^*$ into accepted and rejected. $L_{3Z}$ counts zeros modulo 3.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-DFA-As-String-Classifier/Concept-DFA-As-String-Classifier.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.LangDef        import *
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.AnimateDFA     import *

import jove; print('Jove loaded from', list(jove.__path__)[0])
import jove.AnimateDFA as _a; print('animation toolbar:',
      'ready' if hasattr(_a.AnimateDFA, '_ipython_display_')
      else 'STALE -- restart the runtime, then re-run')

## 1. The idea


The purpose of almost every machine here is to **partition $\Sigma^*$** into accepted
and rejected strings. Viewed that way a DFA is a **string classifier** &mdash; a total
decision procedure, which is only coherent because $\delta$ is total.

$L_{3Z} = \{w : \#_0(w) \bmod 3 = 0\}$. The machine never *counts* the zeros; it tracks
$\#_0 \bmod 3$ &mdash; three states for three residues.

## 2. Definitions

### $L_{3Z}$: zeros divisible by three

In [ ]:
L3Z = md2mc('''DFA
IF : 0 -> S1
IF : 1 -> IF
S1 : 0 -> S2
S1 : 1 -> S1
S2 : 0 -> IF
S2 : 1 -> S2
''')
print("states :", sorted(L3Z["Q"]), " (one per residue mod 3)")

### The classifier view: split $\Sigma^*$ in two

In [ ]:
from itertools import product

def classify(D, n):
    acc, rej = [], []
    for k in range(n+1):
        for p in product(sorted(D["Sigma"]), repeat=k):
            s = ''.join(p)
            (acc if accepts_dfa(D, s) else rej).append(s)
    return acc, rej

<!-- nav-strip -->

---

&larr;&nbsp;[Ch4&nbsp;7.&nbsp;The Language of a DFA, and the Definition of a Regular Language](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Language-Of-A-DFA/Concept-Language-Of-A-DFA.ipynb) &nbsp;&middot;&nbsp; [**Chapter 4** index](https://github.com/ganeshutah/Jove/blob/master/Chapter4-DFA/README.md) &nbsp;&middot;&nbsp; [Ch4&nbsp;9.&nbsp;Basics of Designing a DFA](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4-DFA/Concept-Designing-A-DFA/Concept-Designing-A-DFA.ipynb)&nbsp;&rarr;

---

## 3. Tests

The machine agrees with the arithmetic specification.

In [ ]:
ok = True
for k in range(7):
    for p in product('01', repeat=k):
        s = ''.join(p)
        if accepts_dfa(L3Z, s) != (s.count('0') % 3 == 0): ok = False
print("DFA matches (#0 mod 3 == 0) on all strings up to length 6 :", ok)
assert ok

The partition of $\Sigma^*$, up to length 3.

In [ ]:
acc, rej = classify(L3Z, 3)
print("accepted (%d) :" % len(acc), acc[:10])
print("rejected (%d) :" % len(rej), rej[:10])
assert len(acc) + len(rej) == sum(2**k for k in range(4))
print("\nEvery string lands in exactly one bucket -- that is what 'total' means.")

Three states handle **arbitrarily many** zeros, because a residue is all you keep.

In [ ]:
long0 = '0' * 300
print("300 zeros accepted?", accepts_dfa(L3Z, long0), " (300 mod 3 == 0)")
assert accepts_dfa(L3Z, long0)

## 4. Animation

Watch the machine cycle through the three residues. It is counting modulo 3, never counting outright.

In [ ]:
from jove.AnimateDFA import *
AnimateDFA(L3Z, FuseEdges=True)

## 5. Exercises


1. Build the analogous machine for "zeros divisible by 4".
2. Why can a DFA track $\#_0 \bmod 3$ but not $\#_0$ itself?
3. Express $L_{3Z}$ as a set comprehension, then check it against the machine.

In [ ]:
# Your work for the exercises above.

## 6. Where next

In [ ]:
# Previous / next, and a search box for all 253 concepts.
# Type a chapter (Chapter7, ch7, NFA) or words from a title (pumping).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:
#     load_here('Chapter7-NFA/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter4-DFA/Concept-DFA-As-String-Classifier')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')